# 4 · Wing and tail sizing, and closing the design

VLM lift slope with the Göthert rule, the drag build-up, and the closed sizing loop.
Lesson: [04](../docs/lessons/04-wing-and-tail-sizing.md) ·
Theory: [wing and tail sizing](../docs/wing_tail_sizing.md).

In [1]:
import sys
import numpy as np
import matplotlib.pyplot as plt
from acdesign import plotting as P
from acdesign.requirements import B737_800_MISSION as M
from acdesign.units import FT, G0, LB, NMI

P.use_style()
np.set_printoptions(precision=4, suppress=True)
sys.path.insert(0, "../validation")  # cited reference data and the reference aircraft
from reference import b737_800

R = b737_800()
from acdesign.external.wing_aero import Planform, lift_slope, lift_slope_datcom, downwash_gradient
from acdesign.sizing import size_aircraft

In [2]:
w = Planform(124.6, 9.44, 0.278, np.deg2rad(25))
for Mach in (0.0, 0.5, 0.78):
    print(f"Mach {Mach}: VLM {lift_slope(w, Mach):.3f}/rad, DATCOM {lift_slope_datcom(w, Mach):.3f}/rad")
print(f"downwash gradient at the tail (17.68 m aft, 1.5 m up), Mach 0.78: {downwash_gradient(w, 17.68, 1.5, 0.78):.3f}")

Mach 0.0: VLM 4.691/rad, DATCOM 4.789/rad
Mach 0.5: VLM 5.162/rad, DATCOM 5.272/rad
Mach 0.78: VLM 6.247/rad, DATCOM 6.362/rad
downwash gradient at the tail (17.68 m aft, 1.5 m up), Mach 0.78: 0.430


In [3]:
a = size_aircraft(M)
print(f"W0 {a.W0:,.0f} kg | OEW {a.oew:,.0f} | fuel {a.fuel_mass:,.0f} | S {a.S:.1f} m2 | T {a.thrust / 1e3:.1f} kN")
print(f"CD0 {a.polar.CD0 * 1e4:.0f} counts, e {a.polar.e:.3f}, cruise L/D {a.fuel.LD_cruise:.2f}")
for k, v in a.drag_breakdown.items():
    print(f"  {k:26s} {v * 1e4:5.1f} counts")

W0 69,962 kg | OEW 33,661 | fuel 19,972 | S 115.1 m2 | T 97.8 kN
CD0 180 counts, e 0.810, cruise L/D 17.65
  wing                        61.4 counts
  horizontal tail             17.7 counts
  vertical tail               13.2 counts
  fuselage                    65.7 counts
  nacelles                    16.4 counts
  leakage and protuberances    5.2 counts


Change a design choice — a 30° sweep — and re-size:

In [4]:
from acdesign.sizing import Design

b = size_aircraft(M, Design(sweep=np.deg2rad(30)))
print(f"W0 {b.W0:,.0f} kg ({b.W0 / a.W0 - 1:+.1%}), fuel {b.fuel_mass:,.0f} kg ({b.fuel_mass / a.fuel_mass - 1:+.1%})")

W0 70,425 kg (+0.7%), fuel 20,028 kg (+0.3%)
